# End-to-end Test, adding smoothing and onset detection

## First, load the model

In [1]:
import torch
import torch.nn.functional as F
import torch.nn as nn

class ChordCNN(nn.Module):
    def __init__(self, num_classes: int = 26):
        super().__init__()
 
        # Conv backbone: extracts chord features from chroma spectrogram
        # Block 1: (B,1,12,15) → (B,32,12,15)  — learns interval patterns
        # Block 2: (B,32,12,15) → (B,64,6,7)   — learns full chord shapes

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),     # 320 params
            nn.BatchNorm2d(32),                              # 64 params
            nn.ReLU(),
            nn.Dropout2d(0.15),
 
            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),    # 18,496 params
            nn.BatchNorm2d(64),                              # 128 params
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                              # (12,15) → (6,7)
            nn.Dropout2d(0.15),
        )
 
        # Global average pool: (B,64,6,7) → (B,64)
        self.gap = nn.AdaptiveAvgPool2d(1)
 
        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(64, 32),                               
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes),                   
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:   
        x = self.features(x)
        x = self.gap(x).flatten(1)
        x = self.classifier(x)
        return x
    
    @torch.no_grad()
    def predict(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        self.eval()
        logits = self.forward(x)
        probs = F.softmax(logits, dim=1)
        return probs.argmax(dim=1), probs

def build_model(num_classes: int = 26) -> ChordCNN:
    """Create model with Kaiming initialization."""
    model = ChordCNN(num_classes)
    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)
    return model

ModuleNotFoundError: No module named 'torch'

In [33]:
model = build_model(num_classes=26)
checkpoint = torch.load('../models/checkpoints/best_chord_cnn.pth', weights_only=True)
model.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

## Smoothing

In [16]:
from scipy.stats import mode
from numpy.lib.stride_tricks import sliding_window_view

HOP_LENGTH = 512

def smooth_predictions(predictions, vote_window=5):
    half = vote_window // 2
    padded_predictions = np.pad(predictions, half, mode="edge")
    sliding_view = sliding_window_view(padded_predictions, vote_window)
    smoothed_preds = mode(sliding_view, axis=1, keepdims=False).mode
    return smoothed_preds
    
    
    

## Test Smoothing

In [4]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt

def extract_chroma_cqt(source):
    if isinstance(source, str):
        # File path
        waveform, sr = librosa.load(source, sr=22050)
    else:
        audio = source["audio"].get_all_samples()
        waveform = audio.data.cpu().numpy()
        if waveform.ndim == 2:
            waveform = waveform.mean(axis=0)
        else:
            waveform = waveform.squeeze()
        sr = audio.sample_rate

        if sr != 22050:
            waveform = librosa.resample(waveform, orig_sr=sr, target_sr=22050)
            sr = 22050

    y_harmonic = librosa.effects.harmonic(waveform, margin=1.0)

    chroma = librosa.feature.chroma_cqt(
        y=y_harmonic, sr=sr, hop_length=512,
        fmin=32.7, n_chroma=12, bins_per_octave=12
    )

    return chroma, y_harmonic

def slice_into_windows(chroma, context_frames=15):
    if chroma.ndim != 2:
        raise ValueError(f"Expected chroma shape (12, T), got {chroma.shape}")

    n_frames = chroma.shape[1]

    if n_frames < context_frames:
        necessary_padding = context_frames - n_frames
        padded = np.pad(chroma, ((0, 0), (0, necessary_padding)))
        return padded[np.newaxis, :]

    windows = []
    for i in range(n_frames - context_frames + 1):
        window = chroma[:, i:i + context_frames]
        windows.append(window)

    return np.array(windows)

In [5]:
# Need to load the dataset to translate indexes to classes

from datasets import load_dataset

ds = load_dataset("rodriler/isolated-guitar-chords")

CHORD_CLASSES = ds["train"].features["label"].names + ["Bdim"]

/Users/thomascanro/Desktop/CLASSES/CPEDesign/ChordSenseDisplay/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
chroma, y = extract_chroma_cqt("../Datasets/MyRecordings/D_chord.wav")
windows = slice_into_windows(chroma)

# Convert to tensor
features = torch.tensor(np.array(windows), dtype=torch.float32).unsqueeze(1)

# Predict
model.eval()
with torch.no_grad():
    outputs = model(features.float())
    preds = torch.argmax(outputs, dim=1)

# Build a label-to-index map for converting string labels
label_to_idx = {name: i for i, name in enumerate(CHORD_CLASSES)}
noise_idx = label_to_idx["Noise"]

smoothed = smooth_predictions(preds, vote_window=9)
# Show predictions per window
for i, pred in enumerate(smoothed):
    time_sec = i * 512 / 22050
    print(f"{time_sec:.2f}s: {CHORD_CLASSES[pred]}")

0.00s: D
0.02s: D
0.05s: D
0.07s: D
0.09s: D
0.12s: D
0.14s: D
0.16s: D
0.19s: D
0.21s: D
0.23s: D
0.26s: D
0.28s: D
0.30s: D
0.33s: D
0.35s: D
0.37s: D
0.39s: D
0.42s: D
0.44s: D
0.46s: D
0.49s: D
0.51s: D
0.53s: D
0.56s: D
0.58s: D
0.60s: D
0.63s: D
0.65s: D
0.67s: D
0.70s: D
0.72s: D
0.74s: D
0.77s: D
0.79s: D
0.81s: D
0.84s: D
0.86s: D
0.88s: D
0.91s: D
0.93s: D
0.95s: D
0.98s: D
1.00s: D
1.02s: D
1.04s: D
1.07s: D
1.09s: D
1.11s: D
1.14s: D
1.16s: D
1.18s: D
1.21s: D
1.23s: D
1.25s: D
1.28s: D
1.30s: D
1.32s: D
1.35s: D
1.37s: D
1.39s: D
1.42s: D
1.44s: D
1.46s: D
1.49s: D
1.51s: D
1.53s: D
1.56s: D
1.58s: Dm
1.60s: Dm
1.63s: Dm
1.65s: Dm
1.67s: Dm
1.70s: Dm
1.72s: Dm
1.74s: Dm
1.76s: Dm
1.79s: Dm
1.81s: Dm
1.83s: Dm
1.86s: Dm
1.88s: Dm
1.90s: Dm
1.93s: Dm
1.95s: Dm
1.97s: Dm
2.00s: Dm
2.02s: Dm
2.04s: Dm
2.07s: Dm
2.09s: Dm
2.11s: Dm
2.14s: Dm
2.16s: Dm
2.18s: D
2.21s: D
2.23s: D
2.25s: D
2.28s: D
2.30s: D
2.32s: D
2.35s: D
2.37s: D
2.39s: D
2.41s: D
2.44s: D
2.46s: D
2.48s: D
2.

## Now, let's add onset detection for a more accurate result

In [27]:
def final_prediction(
    smoothed_preds, y, noise_idx, sr=22050, hop_len=512, post_onset_offset=2,
    post_onset_length=5, min_segment_frames=4,
    ):
    
    onsets = librosa.onset.onset_detect(y=y, sr=sr, hop_length=hop_len, backtrack=True)
    
    final_pred = np.empty_like(smoothed_preds)
    segments = []  # list of (start_frame, end_frame, label)
    
    current_chord = noise_idx
    current_start = 0
    
    for onset_frame in onsets:
        # read the post-onset window to find out what chord starts here
        lo = onset_frame + post_onset_offset
        hi = min(lo + post_onset_length, len(smoothed_preds))
        if lo >= len(smoothed_preds):
            break  # onset too close to end to read a post-window
        
        post_onset_chord = mode(smoothed_preds[lo:hi], keepdims=False).mode
        
        if post_onset_chord != current_chord:
            # commit a boundary at this onset
            segments.append((current_start, onset_frame, current_chord))
            current_start = onset_frame
            current_chord = post_onset_chord
        # else: same chord, this onset was just another strum — ignore
    
    # close out the final segment
    segments.append((current_start, len(smoothed_preds), current_chord))
    
    # paint frame-level output from segments
    for start, end, label in segments:
        final_pred[start:end] = label
    
    return {
        "frame_labels": final_pred,
        "segments": segments,
        "onset_frames": onsets,
    }
    

In [28]:
final_pred = final_prediction(smoothed, y, noise_idx)

In [29]:
print(final_pred)

{'frame_labels': array([24, 24, 24, 24, 24, 24, 24, 24, 24, 10, 10, 10, 10, 10, 10, 10, 10,
       10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
       10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
       10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
       10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
       10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
       10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10]), 'segments': [(0, np.int64(9), 24), (np.int64(9), 114, np.int64(10))], 'onset_frames': array([9])}


In [30]:
for i, pred in enumerate(final_pred["frame_labels"]):
    time_sec = i * 512 / 22050
    print(f"{time_sec:.2f}s: {CHORD_CLASSES[pred]}")

0.00s: Noise
0.02s: Noise
0.05s: Noise
0.07s: Noise
0.09s: Noise
0.12s: Noise
0.14s: Noise
0.16s: Noise
0.19s: Noise
0.21s: D
0.23s: D
0.26s: D
0.28s: D
0.30s: D
0.33s: D
0.35s: D
0.37s: D
0.39s: D
0.42s: D
0.44s: D
0.46s: D
0.49s: D
0.51s: D
0.53s: D
0.56s: D
0.58s: D
0.60s: D
0.63s: D
0.65s: D
0.67s: D
0.70s: D
0.72s: D
0.74s: D
0.77s: D
0.79s: D
0.81s: D
0.84s: D
0.86s: D
0.88s: D
0.91s: D
0.93s: D
0.95s: D
0.98s: D
1.00s: D
1.02s: D
1.04s: D
1.07s: D
1.09s: D
1.11s: D
1.14s: D
1.16s: D
1.18s: D
1.21s: D
1.23s: D
1.25s: D
1.28s: D
1.30s: D
1.32s: D
1.35s: D
1.37s: D
1.39s: D
1.42s: D
1.44s: D
1.46s: D
1.49s: D
1.51s: D
1.53s: D
1.56s: D
1.58s: D
1.60s: D
1.63s: D
1.65s: D
1.67s: D
1.70s: D
1.72s: D
1.74s: D
1.76s: D
1.79s: D
1.81s: D
1.83s: D
1.86s: D
1.88s: D
1.90s: D
1.93s: D
1.95s: D
1.97s: D
2.00s: D
2.02s: D
2.04s: D
2.07s: D
2.09s: D
2.11s: D
2.14s: D
2.16s: D
2.18s: D
2.21s: D
2.23s: D
2.25s: D
2.28s: D
2.30s: D
2.32s: D
2.35s: D
2.37s: D
2.39s: D
2.41s: D
2.44s: D
2.46s: D
2

In [31]:
print(final_pred["segments"])

[(0, np.int64(9), 24), (np.int64(9), 114, np.int64(10))]


### Test a more Complete Song

In [32]:
chroma, y = extract_chroma_cqt("../Datasets/MyRecordings/test_song.wav")
windows = slice_into_windows(chroma)

features = torch.tensor(np.array(windows), dtype=torch.float32).unsqueeze(1)

# Predict
model.eval()
with torch.no_grad():
    outputs = model(features.float())
    preds = torch.argmax(outputs, dim=1)

# Build a label-to-index map for converting string labels
label_to_idx = {name: i for i, name in enumerate(CHORD_CLASSES)}
noise_idx = label_to_idx["Noise"]

smoothed = smooth_predictions(preds, vote_window=9)

final_pred = final_prediction(smoothed, y, noise_idx)

for i, pred in enumerate(final_pred["frame_labels"]):
    time_sec = i * 512 / 22050
    print(f"{time_sec:.2f}s: {CHORD_CLASSES[pred]}")

0.00s: Noise
0.02s: Noise
0.05s: G
0.07s: G
0.09s: G
0.12s: G
0.14s: G
0.16s: G
0.19s: G
0.21s: G
0.23s: G
0.26s: G
0.28s: G
0.30s: G
0.33s: G
0.35s: G
0.37s: G
0.39s: G
0.42s: G
0.44s: G
0.46s: G
0.49s: G
0.51s: G
0.53s: G
0.56s: G
0.58s: G
0.60s: G
0.63s: G
0.65s: G
0.67s: G
0.70s: G
0.72s: G
0.74s: G
0.77s: G
0.79s: G
0.81s: G
0.84s: G
0.86s: G
0.88s: G
0.91s: G
0.93s: G
0.95s: G
0.98s: G
1.00s: G
1.02s: G
1.04s: G
1.07s: G
1.09s: G
1.11s: G
1.14s: G
1.16s: G
1.18s: G
1.21s: G
1.23s: G
1.25s: G
1.28s: G
1.30s: G
1.32s: G
1.35s: G
1.37s: G
1.39s: G
1.42s: G
1.44s: G
1.46s: G
1.49s: G
1.51s: G
1.53s: G
1.56s: G
1.58s: G
1.60s: G
1.63s: G
1.65s: G
1.67s: G
1.70s: G
1.72s: G
1.74s: G
1.76s: G
1.79s: G
1.81s: G
1.83s: G
1.86s: G
1.88s: G
1.90s: G
1.93s: G
1.95s: G
1.97s: G
2.00s: G
2.02s: G
2.04s: G
2.07s: G
2.09s: G
2.11s: G
2.14s: G
2.16s: G
2.18s: G
2.21s: Noise
2.23s: Noise
2.25s: Noise
2.28s: Noise
2.30s: Noise
2.32s: Noise
2.35s: Noise
2.37s: Noise
2.39s: Noise
2.41s: Noise
2.44s: 